# Optimizing Agentic AI Collaboration v1.2.0 - GSM8K Full Workflow Template

## Project aims

 1. **a LangChain-based multi-agent Society-of-Minds prototype** supporting **≥3 interaction protocols**,
 2. **benchmark gains** (≥5% on **GSM8K** and **MMLU** vs a single-agent SLM baseline), and
 3. a **cost/benefit profile** using **latency + total token consumption**, all tracked in **W&B** for a claim-to-evidence table.

## Current notebook version achievements

The code acts as a **template** for execution and evaluation of both single agent baselines and multi-agentic systems over the **GSM8K** dataset. The run is orchestrated and stored through W&B.
It covers the full workflow of the project experiments:
1. Automated ingestion of GSM8K dataset and creation of single/multi agent pipeline.
2. Evaluation of the agentic system over multiple GSM8K examples. Each sample is evaluated with 3 different seeds to reduce estimation variability.
3. Per-step and aggregated logging of **model's correctness/accuracy**, **latency**, **token consumption** (output and total), and **GPU resource usage** (peak VRAM in MB), plus generation throughput in tok/s as a control metric. Correctness, latency and peak VRAM matrices are persisted to a W&B artifact for downstream statistical analysis. Those metrics are measured for both a multi-agent system and its single-agent baseline.
4. Definition of statistically consistent estimators and confidence intervals for accuracy, latency, output and total token consumption, and GPU VRAM usage, for both the MAS and the baseline separately and altogether. The statistical measures are stored on W&B.

It could produce evidence for the thesis' central claims (benchmark gains + compute trade-off on GSM8K/MMLU) if the implemented agentic systems were created from more recent state-of-the-art SLMs, included more complex multi-agent protocols, and were evaluated with more data samples.

## Relation with the project WPs

**WP2 - T1: End-to-end pipeline template definition.** It defines the full end-to-end pipeline (GSM8K/MMLU data ingestion → LCEL generation → evaluation → W&B logging) that the WP2 baselines and the L1/L2 multi-agent experiments will use as a template.


## Setup

**Hardware target:** Nvidia T4 GPU (on Kaggle or Colab free tier).
**Model:** `mistralai/Mistral-7B-Instruct-v0.2` with 4-bit NF4 quantization.

---
### Additional information

1. **LangChain-based multi-agent orchestration** — agents are defined as LangChain `LLMChain` instances with structured `PromptTemplate`s, composed via a custom sequential pipeline. This replaces ad-hoc string concatenation with a modular, extensible framework.
2. **Weights & Biases (W&B) experiment tracking** — every agent call, its latency, token counts, VRAM usage, throughput and full text are logged as W&B artifacts and metrics, enabling richer post-hoc analysis across runs.

---

### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel.
1. In **Settings**, enable **Accelerator: GPU T4 x2**; the notebook uses only `cuda:0`.
2. Add and enable `GITHUB_TOKEN`, `HF_TOKEN`, and `WANDB_API_KEY` under **Add-ons → Secrets**.
3. Click **Run All**.

### Before running (Colab)

Complete these manual steps in Colab:

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What the current notebook implements

The implementation is a **LangChain refactor** of a *single* 3-step pipeline:

* **Proposer → Critic → Refiner**, plus a **single-agent baseline**.
* **W&B logging** of per-step latency, output length, input/output token counts, GPU peak VRAM, and generation throughput; plus a W&B `Table` comparing outputs across tasks.
* Evaluation is on **GSM8K random samples.**
**Observed latency, token consumption, VRAM usage and their overhead (from the notebook run outputs)**. **Accuracy, latency, token consumption and VRAM usage averages are estimated with both a point and a 95% confidence interval**
* Multiple random seeds are defined to **allow reproducibility**.


---

## Objective-by-objective diagnosis

### OBJ-1 (multi-agent prototype, ≥3 protocols)

* **Partially relevant**: there is a working multi-agent scaffold (roles, orchestration, logging).
* **Not compliant** with the "≥3 distinct interaction protocols" requirement: there is **only one protocol** (Critique→Refine). No majority voting, no self-consistency, no multi-round debate.

### OBJ-2 (≥5% accuracy gain on GSM8K + MMLU vs single-agent baseline)

* **Addressed**: dataset harness, scoring based on accuracy (with both point and interval-based estimations) compared to a single-agent baseline. Execution repeated with 3 distinct seeds to gain consistency and reduce estimations variability.

### OBJ-3 (accuracy-to-compute trade-off: latency + total token consumption)

* **Addressed**: latency is measured and logged. Token consumption (input and output tokens, separated) is now tracked per-step and aggregated for MAS vs baseline. GPU peak VRAM and generation throughput (tok/s) are also collected per sample; peak VRAM is persisted to the W&B artifact.


### Scope alignment

* Model is **7B** and 4-bit quantized (within the "1B–7B" envelope), so *model class is in-scope*.
* The current artifact is able to compare with statistically consistent measures of the computational cost and the accuracy the model's execution. It logs all the executions through WandB.
* However, the model and the MAS used remain quite far away from the current state-of-the-art. Furthermore, only 20 distinct MMLU samples are evaluated, *not enough for a consistent, low-variance estimation of the agents performance*.

## 1. Setup and Installation

We install all the libraries required for the Github repo.
For loading models and data, orchestrating agents, and saving the results, we use Hugging Face stack (same as v0.4), plus `langchain`, `langchain-community`, `langchain-core`, `datasets` and `wandb`.
If we are running from Colab, we import the Github repo, which includes utils python files with helper methods.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"   
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [2]:
# Install the project dependencies in the selected runtime
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: langchain-core
    Found existing installation: la

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.


In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Define Run Metadata

We define the W&B project and configuration metadata, in order to later create a W&B run that will store all the agent calls, metrics and statistics of the agents' execution.


In [ ]:
RUN_NAME = "v1.2.0-GSM8K-Final-Template--Mistral-7B-Instruct"

PROJECT = "agentic-collab-setup" if (ENVIRONMENT == "kaggle") else "dev-agentic-collab-setup"
print(f"W&B Project: {PROJECT}")

CONFIG = {
        "model_id":       "mistralai/Mistral-7B-Instruct-v0.2",
        "quantization":   "nf4-4bit",
        "framework":      "langchain",
        "agents":         ["proposer", "critic", "refiner"],
        "max_new_tokens": {"proposer": 1024, "critic": 1024, "refiner": 1024, "baseline": 1024},
        "temperature":    0.7,
        "top_p":          0.95,
        "top_k":          50,
        "eval_dataset":   "gsm8k",
        "eval_size":      20,
        "data_seed":      42,
        "generation_seeds": [42, 123, 456],
    }

W&B Project: agentic-collab-setup


## 4. Data Loading
We load the GSM8K (Grade School Math tasks) dataset, which will be used later to evaluate the agentic workflows.


In [5]:
from utils.data import load_gsm8k

# ── GSM8K DATASET ──────────────────────────────────────────────────
gsm8k_dataset = load_gsm8k(split="test")

print(f"✅ GSM8K dataset loaded with {len(gsm8k_dataset)} examples.")
print(f"Example question: {gsm8k_dataset[0]['question']}")
print(f"Example answer:  {gsm8k_dataset[0]['answer']}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ GSM8K dataset loaded with 1319 examples.
Example question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Example answer:  Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## 5. Model Loading (4-bit Quantized)

We load `Mistral-7B-Instruct-v0.2` in NF4 4-bit via `bitsandbytes`, which keeps VRAM usage under ~6 GB on a T4.


In [6]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = CONFIG["model_id"]
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✅ Model loaded on cuda:0 — dtype torch.bfloat16


## 6. Defining Agents as LangChain Chains

Each agent is a `LLMChain` built from a `PromptTemplate`.  This is a clean separation of *role definition* (the template) from *execution engine* (the LLM), making it trivial to swap models, add agents, or modify prompts independently.


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Parser
parser = StrOutputParser()

# ── Proposer ──────────────────────────────────────────────────
proposer_template = PromptTemplate.from_template(
        "You are the Proposer Agent. Analyze the following task and "
        "provide a clear, step-by-step solution.\n\n"
        "{task}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{{}}."
)

# ── Critic ────────────────────────────────────────────────────
critic_template = PromptTemplate.from_template(
        "You are the Critic Agent. Your role is to meticulously "
        "review the proposed solution for the given task. Identify "
        "any potential flaws, logical errors, missed assumptions, "
        "or areas for improvement. Do not provide a new solution, "
        "only critique the existing one.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution to review:\n---\n{proposal}\n---\n\n"
        "Think step by step."
        "Provide your critical analysis below:"
)

# ── Refiner ───────────────────────────────────────────────────
refiner_template = PromptTemplate.from_template(
        "You are the Refiner Agent. Produce a final, correct solution "
        "to a task using the initial proposal and critique below. Carefully "
        "consider the critique and fix any flaws in the original "
        "proposal.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution:\n---\n{proposal}\n---\n\n"
        "Critique of the Solution:\n---\n{critique}\n---\n\n"
        "Please reason step by step, and put your final answer, based on the critique, within \\boxed{{}}."
)


# ── Single-Agent Baseline ────────────────────────────────────
baseline_template = PromptTemplate.from_template(
        "{task}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{{}}."
)

print("✅ All agent chains defined.")

✅ All agent chains defined.


## 7. Multi-Agent Orchestrator with W&B Logging

**v1.2.x additions:** `run_agent_step` now counts output and total tokens (using the model's own tokenizer, outside the timed region so latency measurements stay uncontaminated). `run_multi_agent` wraps the MAS block (Proposer → Critic → Refiner) and the baseline block independently with GPU peak-VRAM measurement (`torch.cuda`). Throughput (output tok/s) is derived from already-collected token counts and latencies — no additional overhead. All new metrics are logged to W&B per sample and accumulated into matrices for the W&B artifact (except for Throughput, which is used as a "control" metric and will not be taken into account into the analysis).

In [8]:
import time
from utils.logging_utils import log_metrics
from utils.instrumentation import (
    count_tokens, render_prompt,
    reset_gpu_peak, gpu_peak_mem_mb,
    tokens_per_second,
)

def run_agent_step(chain, inputs, step_name, prompt_template):
    """Run a single LangChain chain, time it, count tokens, measure per-step peak VRAM, and log to W&B.

    Token counting and VRAM measurement are performed outside the timed region
    so the latency measurement wraps only ``chain.invoke``.

    Returns
    -------
    tuple[str, float, int, int, float, float]
        (output_text, elapsed_s, input_tokens, output_tokens, peak_vram_mb, tps)
    """
    # Input tokens: counted BEFORE the timed region
    prompt_str   = render_prompt(prompt_template, inputs)
    input_tokens = count_tokens(tokenizer, prompt_str)

    # Reset peak VRAM counter for this step's independent measurement
    reset_gpu_peak()

    # Timed region: pure chain.invoke only
    t0          = time.perf_counter()
    output_text = chain.invoke(inputs)
    elapsed     = time.perf_counter() - t0

    # Post-invoke measurements: outside the timed region
    output_tokens = count_tokens(tokenizer, output_text)
    total_tokens  = input_tokens + output_tokens
    peak_vram_mb  = gpu_peak_mem_mb()
    tps           = tokens_per_second(output_tokens, elapsed)

    log_metrics({
        f"{step_name}/latency_s":    round(elapsed, 2),
        f"{step_name}/output_tokens":output_tokens,
        f"{step_name}/total_tokens": total_tokens,
        f"{step_name}/peak_vram_mb": round(peak_vram_mb, 2),
        f"{step_name}/throughput_tps": round(tps, 2)
    })

    print(
        f"  ⏱  {step_name}: {elapsed:.1f}s  |  "
        f"{len(output_text)} chars  |  "
        f"in={input_tokens} out={output_tokens} tok  |  "
        f"peak={peak_vram_mb:.0f} MB  |  {tps:.1f} tok/s"
    )
    return output_text, elapsed, input_tokens, output_tokens, peak_vram_mb


def run_multi_agent(task, task_label="task", eval_type=None):
    """
    Execute the full Proposer → Critic → Refiner pipeline plus a
    single-agent baseline, with W&B logging of latency, tokens, GPU
    peak VRAM (per-agent + MAS aggregate), GPU utilization %, and throughput.

    Peak VRAM is measured independently per agent step (reset before each
    chain.invoke via run_agent_step). The MAS aggregate is max(proposer,
    critic, refiner) — correct for sequential execution.

    Returns
    -------
    dict with keys:
        proposal, critique, refined, baseline  (text outputs)
        t_prop, t_crit, t_ref, t_base          (per-step latencies, s)
        out_prop, out_crit, out_ref            (per-agent output tokens)
        agent_in_tok, agent_out_tok            (MAS aggregated tokens)
        agent_total_tok                        (MAS total tokens in+out)
        base_in_tok, base_out_tok, base_total_tok  (baseline tokens)
        peak_prop, peak_crit, peak_ref         (per-agent peak VRAM, MB)
        agent_peak_mb, base_peak_mb            (aggregate peak VRAM, MB)
        agent_throughput, base_throughput      (aggregate tok/s)
    """
    print(f"\n{'='*60}")
    print(f"🤖 MULTI-AGENT PIPELINE  —  {task_label}")
    print(f"{'='*60}")

    # ── MAS block: Proposer → Critic → Refiner ─────────────────────

    # Step 1 — Proposer
    print("\n[Step 1] Proposer Agent")
    proposal, t_prop, in_prop, out_prop, peak_prop = run_agent_step(
        proposer_chain, {"task": task},
        f"{task_label}/proposer", proposer_template,
    )
    if eval_type is None:
        print(f"\n[Proposer Output]:\n{proposal[:500]}\n{'...' if len(proposal)>500 else ''}")

    # Step 2 — Critic
    print("\n[Step 2] Critic Agent")
    critique, t_crit, in_crit, out_crit, peak_crit = run_agent_step(
        critic_chain, {"task": task, "proposal": proposal},
        f"{task_label}/critic", critic_template,
    )
    if eval_type is None:
        print(f"\n[Critic Output]:\n{critique[:500]}\n{'...' if len(critique)>500 else ''}")

    # Step 3 — Refiner
    print("\n[Step 3] Refiner Agent")
    refined, t_ref, in_ref, out_ref, peak_ref = run_agent_step(
        refiner_chain,
        {"task": task, "proposal": proposal, "critique": critique},
        f"{task_label}/refiner", refiner_template,
    )
    if eval_type is None:
        print(f"\n[Refiner Output]:\n{refined[:500]}\n{'...' if len(refined)>500 else ''}")


    # ── Baseline block ─────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")

    baseline, t_base, in_base, out_base, peak_base = run_agent_step(
        baseline_chain, {"task": task},
        f"{task_label}/baseline", baseline_template,
    )
    if eval_type is None:
        print(f"\n[Baseline Output]:\n{baseline[:500]}\n{'...' if len(baseline)>500 else ''}")

    # ── Aggregate tokens and metrics ────────────────────────────────
    agent_in_tok    = in_prop   + in_crit   + in_ref
    agent_out_tok   = out_prop  + out_crit  + out_ref
    agent_total_tok = agent_in_tok + agent_out_tok
    base_total_tok  = in_base  + out_base
    total_pipeline  = t_prop   + t_crit    + t_ref

    # MAS aggregate peak: max across sequential agents (exact for sequential execution)
    agent_peak_mb    = max(peak_prop, peak_crit, peak_ref)
    agent_throughput = tokens_per_second(agent_out_tok, total_pipeline)
    base_throughput  = tokens_per_second(out_base,       t_base)

    # ── Log per-pipeline summary to W&B ────────────────────────────
    log_metrics({
        # Latency
        f"{task_label}/total_pipeline_latency_s":  round(total_pipeline, 2),
        f"{task_label}/baseline_latency_s":        round(t_base, 2),
        f"{task_label}/latency_overhead_ratio":    round(total_pipeline / max(t_base, 1e-9), 4),
        # Output tokens
        f"{task_label}/agents_output_tokens":      agent_out_tok,
        f"{task_label}/baseline_output_tokens":    out_base,
        f"{task_label}/output_tokens_overhead_ratio": round(agent_out_tok / max(out_base, 1), 4),
        # Total tokens (input + output)
        f"{task_label}/agents_total_tokens":       agent_total_tok,
        f"{task_label}/baseline_total_tokens":     base_total_tok,
        f"{task_label}/total_tokens_overhead_ratio": round(agent_total_tok / max(base_total_tok, 1), 4),
        # GPU — peak VRAM MAS aggregate + baseline + delta
        f"{task_label}/agents_gpu_peak_mb":        round(agent_peak_mb, 2),
        f"{task_label}/baseline_gpu_peak_mb":      round(peak_base, 2),
        f"{task_label}/vram_overhead_diff_mb":     round(agent_peak_mb - peak_base, 2),
        # Throughput
        f"{task_label}/agents_throughput_tps":     round(agent_throughput, 2),
        f"{task_label}/baseline_throughput_tps":   round(base_throughput, 2)
    })

    return {
        "proposal": proposal, "critique": critique,
        "refined":  refined,  "baseline": baseline,
        "t_prop": t_prop, "t_crit": t_crit,
        "t_ref":  t_ref,  "t_base": t_base,
        # Per-agent output tokens
        "out_prop": out_prop, "out_crit": out_crit, "out_ref": out_ref,
        # Aggregated tokens
        "agent_in_tok":    agent_in_tok,    "agent_out_tok":   agent_out_tok,
        "agent_total_tok": agent_total_tok,
        "base_in_tok":     in_base,         "base_out_tok":    out_base,
        "base_total_tok":  base_total_tok,
        # Per-agent peak VRAM
        "peak_prop": peak_prop, "peak_crit": peak_crit, "peak_ref": peak_ref,
        # Aggregate peak VRAM
        "agent_peak_mb": agent_peak_mb, "base_peak_mb": peak_base,
        # Aggregate throughput
        "agent_throughput": agent_throughput,
        "base_throughput":  base_throughput,
    }

print("✅ Orchestrator ready.")


✅ Orchestrator ready.


## 8. Model warmup

Before measuring the agentic pipeline, we run two unmeasured dummy generations to trigger CUDA kernel compilation and autotuning. This prevents one-time startup costs from biasing the reported latency.

In [9]:
# ── Warmup ────────────────────────────────────────────────────────────
# Run a couple of throwaway generations through the same pipeline the
# evaluation uses, so one-time CUDA / bitsandbytes kernel compilation and
# cuBLAS autotuning happen here — not inside the first timed sample.
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=CONFIG["temperature"],
    top_p=CONFIG["top_p"], top_k=CONFIG["top_k"],
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'temperature', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔥 Warming up model (2 iterations, max_new_tokens=256)…


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Warmup complete in 25.9s — first evaluated sample will no longer pay cold-start cost.


25.862654604000454

## 9. Initialize Weights & Biases

We create a W&B run that will persist for the entire notebook session.  All agent calls, latencies, and outputs will be logged here.

In [10]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

run = init_wandb(
    run_name=RUN_NAME,
    project=PROJECT,
    config=CONFIG
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260608_191654-k81cd3qe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run v1.2.0-GSM8K-Final-Template--Mistral-7B-Instruct
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup/runs/k81cd3qe


✅ W&B run: v1.2.0-GSM8K-Final-Template--Mistral-7B-Instruct  —  https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup/runs/k81cd3qe


## 10. GSM8K Evaluation

In [11]:
from utils.data import extract_gsm8k_answer
from utils.evaluation import extract_gsm8k_prediction, gsm8k_match

In [12]:
import warnings
warnings.filterwarnings("ignore") # to ignore transformer warnings that can clutter the output during evaluation

import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

# Fix the evaluation subset once — all seed runs evaluate the same questions
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(gsm8k_dataset), sample_size)

# ── Correctness & latency matrices: rows = problems, cols = seeds ───────
agent_correctness_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
agent_time_matrix           = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)

# ── Token matrices ───────────────────────────────────────────────────────
agent_input_tok_matrix      = np.zeros((sample_size, len(generation_seeds)), dtype=int)
agent_output_tok_matrix     = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_input_tok_matrix   = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_output_tok_matrix  = np.zeros((sample_size, len(generation_seeds)), dtype=int)

# ── GPU matrices ─────────────────────────────────────────────────────────
agent_gpu_peak_matrix       = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_gpu_peak_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=float)

# ── Single unified results table accumulated across ALL seeds ─────────────
results_columns = [
    "problem_id", "seed",
    "task",
    "proposer_output", "critic_output", "refiner_output", "baseline_output",
    "ground_truth", "agents_final_output", "baseline_final_output",
    # Latency per agent + MAS total
    "proposer_latency_s", "critic_latency_s", "refiner_latency_s",
    "baseline_latency_s", "agents_total_latency_s",
    # Output tokens per agent + MAS total + baseline
    "proposer_output_tokens", "critic_output_tokens", "refiner_output_tokens",
    "agents_total_output_tokens", "baseline_output_tokens",
    # Total tokens (input + output) for MAS and baseline
    "agents_total_tokens", "baseline_total_tokens",
    # Peak VRAM
    "agents_gpu_peak_mb", "baseline_gpu_peak_mb",
    # Throughput
    "agents_throughput_tok_s", "baseline_throughput_tok_s",
]
results_rows = []

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    # Rebuild LLM and all agent chains with the current generation seed
    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens["proposer"],
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    proposer_chain = proposer_template | llm | parser
    critic_chain   = critic_template   | llm | parser
    refiner_chain  = refiner_template  | llm | parser
    baseline_chain = baseline_template | llm | parser

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"GSM8K seed={gen_seed} ({sample_size} samples)")):
        task              = item['question']
        ground_truth_full = item['answer']
        true_answer       = extract_gsm8k_answer(ground_truth_full)

        output = None
        try:
            output            = run_multi_agent(task, task_label='gsm8k_sample', eval_type='gsm8k')
            agent_response    = output['refined']
            agent_answer      = extract_gsm8k_prediction(agent_response)
            baseline_response = output['baseline']
            baseline_answer   = extract_gsm8k_prediction(baseline_response)
        except Exception as e:
            print(f"Pipeline failed: {e}")
            agent_answer = baseline_answer = None
            agent_response = baseline_response = str(e)
            if output is None:
                output = {
                    'proposal': '', 'critique': '',
                    'refined':  agent_response, 'baseline': baseline_response,
                    't_prop': 0.0, 't_crit': 0.0, 't_ref': 0.0, 't_base': 0.0,
                    'out_prop': 0, 'out_crit': 0, 'out_ref': 0,
                    'agent_in_tok': 0,   'agent_out_tok': 0,  'agent_total_tok': 0,
                    'base_in_tok':  0,   'base_out_tok':  0,  'base_total_tok':  0,
                    'peak_prop': 0.0, 'peak_crit': 0.0, 'peak_ref': 0.0,
                    'agent_peak_mb': 0.0, 'base_peak_mb': 0.0,
                    'agent_throughput': 0.0, 'base_throughput': 0.0,
                }

        agent_correct    = gsm8k_match(agent_answer,    true_answer)
        baseline_correct = gsm8k_match(baseline_answer, true_answer)
        mas_latency      = output['t_prop'] + output['t_crit'] + output['t_ref']

        # ── Correctness & latency ─────────────────────────────────────
        agent_correctness_matrix[item_idx, run_idx]    = int(agent_correct)
        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        agent_time_matrix[item_idx, run_idx]           = mas_latency
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']

        # ── Token matrices ────────────────────────────────────────────
        agent_input_tok_matrix[item_idx, run_idx]      = output['agent_in_tok']
        agent_output_tok_matrix[item_idx, run_idx]     = output['agent_out_tok']
        baseline_input_tok_matrix[item_idx, run_idx]   = output['base_in_tok']
        baseline_output_tok_matrix[item_idx, run_idx]  = output['base_out_tok']

        # ── GPU matrices ──────────────────────────────────────────────
        agent_gpu_peak_matrix[item_idx, run_idx]       = output['agent_peak_mb']
        baseline_gpu_peak_matrix[item_idx, run_idx]    = output['base_peak_mb']

        log_metrics({
            "gsm8k_sample/agents_accuracy":   int(agent_correct),
            "gsm8k_sample/baseline_accuracy": int(baseline_correct),
        })

        # Accumulate one row for the unified results table (problem_id, seed first)
        results_rows.append([
            item_idx, gen_seed,
            task,
            output['proposal'][:2000], output['critique'][:2000],
            output['refined'][:2000],  output['baseline'][:2000],
            true_answer, agent_answer, baseline_answer,
            # Latency
            round(output['t_prop'], 2), round(output['t_crit'], 2),
            round(output['t_ref'],  2), round(output['t_base'], 2),
            round(mas_latency, 2),
            # Per-agent output tokens + MAS total + baseline
            output['out_prop'], output['out_crit'], output['out_ref'],
            output['agent_out_tok'], output['base_out_tok'],
            # Total tokens
            output['agent_total_tok'], output['base_total_tok'],
            # GPU
            round(output['agent_peak_mb'],  2), round(output['base_peak_mb'],  2),
            # Throughput
            round(output['agent_throughput'], 4), round(output['base_throughput'], 4),
        ])

# ── Log the unified results table ONCE, sorted by problem_id then seed ─────
results_rows.sort(key=lambda r: (r[0], r[1]))
results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({"results_comparison": results_table})


  Seed run 1/3  —  generation_seed=42


GSM8K seed=42 (20 samples):   0%|          | 0/20 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 MULTI-AGENT PIPELINE  —  gsm8k_sample

[Step 1] Proposer Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ⏱  gsm8k_sample/proposer: 63.1s  |  2128 chars  |  in=137 out=743 tok  |  peak=4402 MB  |  11.8 tok/s

[Step 2] Critic Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  gsm8k_sample/critic: 27.1s  |  1507 chars  |  in=920 out=305 tok  |  peak=4617 MB  |  11.3 tok/s

[Step 3] Refiner Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  gsm8k_sample/refiner: 22.8s  |  1134 chars  |  in=1232 out=242 tok  |  peak=4873 MB  |  10.6 tok/s

📊 SINGLE-AGENT BASELINE  —  gsm8k_sample
  ⏱  gsm8k_sample/baseline: 29.6s  |  1063 chars  |  in=110 out=347 tok  |  peak=4396 MB  |  11.7 tok/s


## 11. Log Results Table & Finish W&B Run

Push the accumulated comparison table to W&B and close the run.  You can view the full dashboard (latency charts, text comparisons, overhead ratios) at the URL printed when the run was initialized.

In [13]:
import tempfile
import wandb
import numpy as np
from utils.logging_utils import finish_wandb, log_stat_summary
from utils.statistics import (
    accuracy_statistics,
    time_statistics,
    token_statistics,
    vram_statistics,
    overhead_ratio_statistics,
    overhead_difference_statistics,
    print_accuracy_stats_report,
    print_time_report,
    print_token_report,
    print_vram_report,
    print_overhead_ratio_report,
    print_overhead_difference_report,
)

DATASET = "GSM8K"

# ── Derived matrices ─────────────────────────────────────────────────────
agent_total_tok_matrix    = agent_input_tok_matrix  + agent_output_tok_matrix
baseline_total_tok_matrix = baseline_input_tok_matrix + baseline_output_tok_matrix

# ── Accuracy ─────────────────────────────────────────────────────────────
agent_acc_stats    = accuracy_statistics(agent_correctness_matrix)
baseline_acc_stats = accuracy_statistics(baseline_correctness_matrix)
print_accuracy_stats_report(f"MAS Accuracy ({DATASET})", agent_acc_stats)
print_accuracy_stats_report(f"Baseline Accuracy ({DATASET})", baseline_acc_stats)

# ── Latency ──────────────────────────────────────────────────────────────
agent_time_stats    = time_statistics(agent_time_matrix)
baseline_time_stats = time_statistics(baseline_time_matrix)
print_time_report(f"MAS Latency ({DATASET})", agent_time_stats)
print_time_report(f"Baseline Latency ({DATASET})", baseline_time_stats)

# ── Output token counts ───────────────────────────────────────────────────
agent_out_tok_stats    = token_statistics(agent_output_tok_matrix)
baseline_out_tok_stats = token_statistics(baseline_output_tok_matrix)
print_token_report(f"MAS Output Tokens ({DATASET})", agent_out_tok_stats)
print_token_report(f"Baseline Output Tokens ({DATASET})", baseline_out_tok_stats)

# ── Total token counts (input + output) ─────────────────────────────────
agent_total_tok_stats    = token_statistics(agent_total_tok_matrix)
baseline_total_tok_stats = token_statistics(baseline_total_tok_matrix)
print_token_report(f"MAS Total Tokens ({DATASET})", agent_total_tok_stats)
print_token_report(f"Baseline Total Tokens ({DATASET})", baseline_total_tok_stats)

# ── Peak VRAM ─────────────────────────────────────────────────────────────
agent_vram_stats    = vram_statistics(agent_gpu_peak_matrix)
baseline_vram_stats = vram_statistics(baseline_gpu_peak_matrix)
print_vram_report(f"MAS Peak VRAM ({DATASET})", agent_vram_stats)
print_vram_report(f"Baseline Peak VRAM ({DATASET})", baseline_vram_stats)

# ── Overhead ratios (multiplicative: latency, output tokens, total tokens) ──
overhead_latency_stats      = overhead_ratio_statistics(agent_time_matrix,       baseline_time_matrix)
overhead_out_tok_stats      = overhead_ratio_statistics(agent_output_tok_matrix, baseline_output_tok_matrix)
overhead_total_tok_stats    = overhead_ratio_statistics(agent_total_tok_matrix,  baseline_total_tok_matrix)
print_overhead_ratio_report(f"Latency Overhead ({DATASET})",      overhead_latency_stats)
print_overhead_ratio_report(f"Output Token Overhead ({DATASET})", overhead_out_tok_stats)
print_overhead_ratio_report(f"Total Token Overhead ({DATASET})",  overhead_total_tok_stats)

# ── VRAM overhead (additive) ──────────────────────────────────────────────
overhead_vram_stats = overhead_difference_statistics(agent_gpu_peak_matrix, baseline_gpu_peak_matrix)
print_overhead_difference_report(f"Peak VRAM Overhead ({DATASET})", overhead_vram_stats)

# ── W&B summary ──────────────────────────────────────────────────────────
wandb.summary["n_seeds"]           = len(generation_seeds)
wandb.summary["n_problems"]        = sample_size
wandb.summary["total_experiments"] = sample_size * len(generation_seeds)

# MAS: accuracy statistics
wandb.summary["agents/accuracy"]      = round(agent_acc_stats["p_hat"], 6)
log_stat_summary("agents/accuracy_stats", agent_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

# Baseline: accuracy statistics
wandb.summary["baseline/accuracy"] = round(baseline_acc_stats["p_hat"], 6)
log_stat_summary("baseline/accuracy_stats", baseline_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

# Overhead: latency
wandb.summary["latency_overhead_ratio"] = round(overhead_latency_stats["geometric_mean"], 4)
log_stat_summary("latency_overhead_ratio/stats", overhead_latency_stats,
    keys=["arithmetic_mean", "ci_lower", "ci_upper", "between_problem_std"])
# Overhead: output tokens
wandb.summary["output_tokens_overhead_ratio"] = round(overhead_out_tok_stats["geometric_mean"], 4)
log_stat_summary("output_tokens_overhead_ratio/stats", overhead_out_tok_stats,
    keys=["arithmetic_mean", "ci_lower", "ci_upper", "between_problem_std"])
# Overhead: total tokens
wandb.summary["total_tokens_overhead_ratio"] = round(overhead_total_tok_stats["arithmetic_mean"], 4)
log_stat_summary("total_tokens_overhead_ratio/stats", overhead_total_tok_stats,
    keys=["arithmetic_mean", "ci_lower", "ci_upper", "between_problem_std"])
# Overhead: VRAM (additive difference)
wandb.summary["vram_overhead_diff_mb"] = round(overhead_vram_stats["mean_diff"], 2)
log_stat_summary("vram_overhead_difference/stats", overhead_vram_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std"])

## MAS: latency statistics
log_stat_summary("agents/latency", agent_time_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "flat_cv"])
# Baseline: latency statistics
log_stat_summary("baseline/latency", baseline_time_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "flat_cv"])

# MAS: output token statistics
log_stat_summary("agents/output_tokens", agent_out_tok_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])
# Baseline: output token statistics
log_stat_summary("baseline/output_tokens", baseline_out_tok_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])

# MAS: total token statistics
log_stat_summary("agents/total_tokens", agent_total_tok_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])
# Baseline: total token statistics
log_stat_summary("baseline/total_tokens", baseline_total_tok_stats, keys=["mean", "ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])

# MAS: peak VRAM statistics (vram_statistics uses peak_ci_lower / peak_ci_upper)
log_stat_summary("agents/vram", agent_vram_stats,
    keys=["mean_peak", "peak_ci_lower", "peak_ci_upper", "peak_max", "between_problem_peak_std"])
# Baseline: peak VRAM statistics
log_stat_summary("baseline/vram", baseline_vram_stats,
    keys=["mean_peak", "peak_ci_lower", "peak_ci_upper", "peak_max", "between_problem_peak_std"])

# ── Upload all matrices as W&B artifact ──────────────────────────────────
artifact = wandb.Artifact(
    f"evaluation-matrices-{wandb.run.id}",
    type="evaluation-data",
    description=(
        f"All evaluation matrices (N x m) for MAS and baseline. {DATASET} evaluation. "
        "Includes: correctness, execution times, aggregated input/output/total "
        "token counts, GPU peak VRAM (MB)."
    ),
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        # Correctness
        agent_correctness=agent_correctness_matrix,
        baseline_correctness=baseline_correctness_matrix,
        # Execution time
        agent_times=agent_time_matrix,
        baseline_times=baseline_time_matrix,
        # Aggregated token counts
        agent_input_tokens=agent_input_tok_matrix,
        agent_output_tokens=agent_output_tok_matrix,
        agent_total_tokens=agent_total_tok_matrix,
        baseline_input_tokens=baseline_input_tok_matrix,
        baseline_output_tokens=baseline_output_tok_matrix,
        baseline_total_tokens=baseline_total_tok_matrix,
        # GPU peak VRAM
        agent_gpu_peak=agent_gpu_peak_matrix,
        baseline_gpu_peak=baseline_gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print(f"✅ All evaluation matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished. View your dashboard at the URL above.")



  MAS Accuracy (GSM8K)
  Accuracy p̂                  35.00%
  95% Cluster-t CI         [19.42, 50.58]%
  Between-problem std        33.2894%
  ── Supplementary: Wilson ──
  95% Wilson CI              [22.27, 50.29]%
  DEFF                        41.058
  ICC                          0.231
  ── Sample ──
  Problems x Seeds           20 x 3  (n=60)


  Baseline Accuracy (GSM8K)
  Accuracy p̂                  28.33%
  95% Cluster-t CI         [13.77, 42.89]%
  Between-problem std        31.1101%
  ── Supplementary: Wilson ──
  95% Wilson CI              [16.97, 43.33]%
  DEFF                        41.961
  ICC                          0.215
  ── Sample ──
  Problems x Seeds           20 x 3  (n=60)


  MAS Latency (GSM8K)
  Mean (s)                     71.5374
  Std (s)                      24.5250
  CV                            0.3428
  95% BCa CI (s)             [65.8553, 78.8935]  [cluster_bca]
  Between-problem std (s)      15.4858
  ── Percentiles (descriptive) ──
  Median / P50 

wandb: uploading artifact run-k81cd3qe-results_comparison; updating run metadata; uploading artifact evaluation-matrices-k81cd3qe
wandb: uploading artifact evaluation-matrices-k81cd3qe
wandb: 
wandb: Run history:
wandb:         gsm8k_sample/agents_accuracy ▁▁█▁████▁█▁██▁▁▁▁█▁▁▁█▁▁▁█▁▁█▁█▁▁█▁▁▁█▁▁
wandb:      gsm8k_sample/agents_gpu_peak_mb ▁▂▄▃▃▂▂▂▂▃▂▂▂▂▂▂▂▁▃▁▂▂▂▄▂▂▂▂▅▂▄▃▁▃▂▃▁▂▂█
wandb:    gsm8k_sample/agents_output_tokens ▂▄▅▂▃▄▂▂▄▄▃▃▂▃▃▃▂▄▁▄▄▃▂▃▂▂▂▄█▃▅▂▁▂▃▁▄▁▄▅
wandb:   gsm8k_sample/agents_throughput_tps ▄▅▃▄▅▄▅▅▆▇▃▅▇▅▇█▇▄▄▃▄▂▅▃▃▃▆▆▆▇▆▅▄▁▆▅▃▇▆▆
wandb:     gsm8k_sample/agents_total_tokens ▄▅▃▄▂▄▂▂▃▄▃▃▃▃▅▂▄▂▃▃▃▁▃▅█▃▆▅▂▁▂▃▃▄▂▁▃▄▄▇
wandb:      gsm8k_sample/baseline/latency_s ▄▃▄▅▃▂▂▅▄▂▁▂▂▁▃▇█▃▁▁▇▂▃▃▁▂▃▃▄▃▃▃▂▂▃▃▁▁▂▄
wandb:  gsm8k_sample/baseline/output_tokens ▃▄▃▂▂▅▄▂▂▄▂▂▂▁▃▃█▃▁▁▇▂▃▂▃▂▃▁▃▄▄▃▃▄▂▄▂▁▂▁
wandb:   gsm8k_sample/baseline/peak_vram_mb ▂▂▂▃▃▂▂█▃▂▂▂▂▂▅▂▂▃▁▂█▂▃▂▂▂▅▃▂▂▃▁▂▃▂▂▂▂▂▅
wandb: gsm8k_sample/baseline/throughput_tps ▃▄▅▃▅▇▇▅▄▄▄▅▁▇▆█▂▄▄▆▃▄▄▄▅▄▆▅▆▆▆▄▆▅▄▆▄▄▅▄
wandb:   gsm8k_sample/

W&B run finished. View your dashboard at the URL above.


## 12. Performance Analysis & Next Steps

### What W&B gives you

Open the W&B run link to explore:

- **Correctness panel** - baseline and MAS's correctness for each run of each task.
- **Latency panel** — per-agent and total pipeline latency across tasks, showing where time is spent.
- **Token panel** — per-step input/output token counts (`gsm8k_sample/{agent}/total_tokens`, `.../output_tokens`) and aggregated MAS vs baseline totals (`gsm8k_sample/agents_total_tokens`, etc.).
- **Throughput panel** — output tokens per second (`gsm8k_sample/throughput/proposer`, `.../critic`, `.../refiner`, `.../baseline`).
- **GPU panel** — per-sample peak VRAM in MB (`…/agents_gpu_peak_mb`, `…/baseline_gpu_peak_mb`).
- **System tab** — W&B's automatic run-level GPU utilization and memory charts (~15 s cadence, whole-run view, no extra code required).
- **Results table** — unified W&B `Table` with all text outputs plus token, GPU, and throughput columns for every sample.
- **Overhead ratio** — how much more expensive the 3-agent pipeline is vs. the single-shot baseline.

### Notes on measurement methodology

- **Model's answers** are obtained by a regexp from the refiner/baseline's output and compared to the task ground truth to determine correctness.
- **Token counts** are computed with the model's fast tokenizer (`add_special_tokens=False`) on the rendered prompt and raw completion. Both MAS and baseline are measured identically so relative comparisons are sound. Chat-template wrapping may add 1–2 tokens vs what the pipeline actually feeds the model; this systematic offset cancels in MAS-vs-baseline ratios.
- **Peak VRAM** is measured with `torch.cuda.reset_peak_memory_stats()` + `max_memory_allocated()`. The 4-bit weights (~5–6 GB) are resident throughout, so absolute peak reflects weights + KV-cache + activations; the MAS–baseline delta is the informative signal.
- **Throughput** is end-to-end (prefill + decode), not pure decode rate. Both MAS (sum-of-output-tokens / total-MAS-time) and baseline are measured identically.
- **Latency comparability:** absolute latencies in v1.2.x are fully comparable within the same notebook version and posterior ones.

### Next steps

1. **Implementation of robust single-agent baselines:** — run the same pipeline with only the single-agent protocol, with 100-200 GSM8K samples and 3+ distinct random seeds, and using current state-of-the-art SLMs like ``Phi-4-Mini-3.8B``, ``Ministral-3-8B``, ``Gemma-4-E4B`` or ``Qwen-3-8B``, to obtain robsut single-agent baseline accuracies and computational costs (OBJ-2,3).
2. **Protocol expansion** — implement new Level-1 (Multi Agent Verification) protocols, like Majority Voting or Self-Consistency, to meet the ≥3 protocols requirement (OBJ-1).